# 학년별 2PL-IRT 문항모수 추정 (초1 ~ 중3)

초1부터 중3까지 9개 학년의 문항정오답표에 학년마다 2PL 문항반응모형을 적합하고, 모든 문항(`assessmentItemID`)의 변별도 $A$와 난이도 $d$를 하나의 CSV로 저장합니다.

**모형**

$$\mathrm{logit}\,P(x_{ij}=1 \mid \theta_i) = A_j\,\theta_i - d_j,\qquad \theta_i \sim N(0,1)$$

- $A_j > 0$: 변별도입니다. 클수록 능력이 다른 학생을 잘 구분합니다.
- $d_j$: 난이도처럼 해석되는 값입니다. 클수록 어려운 문항입니다. 흔히 쓰는 난이도 $b_j$로 바꾸려면 $b_j = d_j / A_j$입니다.

**추정 방법**

- **marginal MAP(Bayes modal)**: 학생 능력 $\theta$를 Gauss–Hermite 구적법으로 적분해 없앤 주변우도에 아래 prior를 더해서 최대화합니다.
- **결측 처리**: 결측 셀은 각 학생의 우도에서 빠집니다(MAR 가정).
- **prior**: $\log A_j \sim N(0, 0.5^2)$, $d_j \sim N(0, 3^2)$입니다. 정답률이 매우 높고 오답이 몇 개뿐인 문항은 prior가 없으면 $A$, $d$가 발산합니다. prior는 이런 문항의 추정값을 유한하게 묶어 줍니다.
- **학년별 별도 척도**: 학년 사이에 겹치는 문항도 학생도 없어서 척도를 연결할 수 없습니다. 그래서 학년마다 $\theta \sim N(0,1)$ 척도로 따로 추정하고, $A$, $d$는 **같은 학년 안에서만** 비교할 수 있습니다.

## 1. 설정

- `DATA_DIR`, `GRADES`: 데이터 폴더와 학년 목록입니다. 학년마다 `{학년}_문항정오답표.csv`를 읽고, 이 파일은 행이 문항, 열이 학생, 결측이 NaN인 형식입니다.
- `OUT_PATH`: 결과를 저장할 CSV 경로입니다.
- `N_QUAD`: $\theta$ 적분에 쓰는 Gauss–Hermite 구적점의 수입니다.
- `PRIOR_LOGA_SD`, `PRIOR_D_SD`: $\log A$와 $d$에 두는 정규 prior의 표준편차입니다. 값을 키울수록 prior가 약해집니다.

In [1]:
DATA_DIR = r"124.수학분야_학습자_역량_측정"
GRADES = ['초1', '초2', '초3', '초4', '초5', '초6', '중1', '중2', '중3']   # {grade}_문항정오답표.csv: rows = items, columns = students, NaN = missing
OUT_PATH = '2PL_문항모수.csv'
N_QUAD = 41               # Gauss-Hermite quadrature points for theta ~ N(0, 1)
PRIOR_LOGA_SD = 0.5       # prior log A ~ N(0, PRIOR_LOGA_SD^2)
PRIOR_D_SD = 3.0          # prior d ~ N(0, PRIOR_D_SD^2)

## 2. 라이브러리와 데이터 불러오기

`load_grade(grade)`는 한 학년의 정오답표를 읽어서 세 가지를 돌려줍니다.
- 문항 ID 목록
- 응답 행렬 `X`: 학생 × 문항이고, 결측 자리는 0으로 채웁니다.
- 관측 마스크 `mask_obs`: 실제로 응답한 셀만 True입니다.

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

dtype = torch.float64
torch.set_default_dtype(dtype)


# Responses of one grade as students x items (0 where missing) and the observed-cell mask
def load_grade(grade):
    df = pd.read_csv(f'{DATA_DIR}/{grade}_문항정오답표.csv', encoding='utf-8-sig', index_col=0)
    X_raw = torch.tensor(df.values.T, dtype=dtype)
    mask_obs = ~torch.isnan(X_raw)
    X = torch.where(mask_obs, X_raw, torch.zeros_like(X_raw))
    return list(df.index), X, mask_obs

## 3. 2PL 적합 함수

`fit_2pl_mml(X, mask)`는 한 학년의 문항모수를 추정합니다.
- **파라미터화**: $A = \exp(\log A)$로 두어 변별도가 항상 양수가 되게 합니다.
- **목적함수**: 음의 주변로그우도에 $\log A$와 $d$의 prior 항을 더한 값입니다.
- **최적화**: L-BFGS(strong-Wolfe line search)를 쓰고, 목적함수가 더 이상 변하지 않을 때까지 반복합니다.
- **출력**: 최종 목적함수 값과 최대 gradient를 출력합니다. 최대 gradient가 0에 가까우면 수렴한 것입니다.

In [3]:
# 2PL by marginal MAP (Bayes modal; Gauss-Hermite quadrature, theta ~ N(0, 1)): logit P(x_ij = 1) = A_j theta_i - d_j.
#   Priors log A ~ N(0, PRIOR_LOGA_SD^2), d ~ N(0, PRIOR_D_SD^2) keep near-separated items finite (A > 0 by construction).
#   Missing cells are left out of each student's likelihood (MAR).
def fit_2pl_mml(X, mask, n_q=N_QUAD, n_iter=100):
    z, w = np.polynomial.hermite_e.hermegauss(n_q)
    z, logw = torch.tensor(z), torch.log(torch.tensor(w / w.sum()))
    X0, obs = torch.where(mask, X, torch.zeros_like(X)), mask.to(dtype)
    log_A = torch.zeros(X.shape[1], requires_grad=True)
    d = torch.zeros(X.shape[1], requires_grad=True)
    opt = torch.optim.LBFGS([log_A, d], max_iter=n_iter, line_search_fn='strong_wolfe')

    def closure():
        opt.zero_grad()
        lg = z[:, None] * log_A.exp()[None, :] - d[None, :]                                # (Q, J)
        ll = X0 @ F.logsigmoid(lg).T + (obs - X0) @ F.logsigmoid(-lg).T          # (N, Q)
        nll = (-torch.logsumexp(ll + logw, dim=1).sum()
               + 0.5 * (log_A / PRIOR_LOGA_SD).pow(2).sum() + 0.5 * (d / PRIOR_D_SD).pow(2).sum())
        nll.backward()
        return nll

    prev = float('inf')
    for _ in range(20):                      # repeat until the marginal NLL stops changing
        cur = float(opt.step(closure))
        if abs(prev - cur) < 1e-6 * abs(cur):
            break
        prev = cur
    grad = torch.cat([log_A.grad, d.grad]).abs().max().item()
    print(f'2PL marginal MAP: objective {cur:.2f}, max |gradient| {grad:.1e}')
    return log_A.detach().exp(), d.detach()

## 4. 전 학년 적합과 저장

- 모든 학년을 차례로 적합하고, 결과를 `grade, assessmentItemID, A, d` 열로 합쳐 `OUT_PATH`에 저장합니다.
- 마지막에 학년별 $A$와 $d$의 요약 통계를 출력합니다.

In [4]:
# Each grade is fitted on its own scale (no items or students are shared across grades, so the scales are not linked)
results = []
for grade in GRADES:
    item_ids, X, mask_obs = load_grade(grade)
    print(f'[{grade}] {X.shape[0]} students x {X.shape[1]} items, missing rate {(~mask_obs).to(dtype).mean().item():.3f}')
    A_hat, d_hat = fit_2pl_mml(X, mask_obs)
    results.append(pd.DataFrame({'grade': grade, 'assessmentItemID': item_ids, 'A': A_hat.numpy(), 'd': d_hat.numpy()}))

params = pd.concat(results, ignore_index=True)
params.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')
print(f'saved {OUT_PATH} ({len(params)} items)')
print(params.groupby('grade', sort=False)[['A', 'd']].describe().round(3))

[초1] 2566 students x 943 items, missing rate 0.879
2PL marginal MAP: objective 116228.27, max |gradient| 4.0e-05
[초2] 2798 students x 1022 items, missing rate 0.896
2PL marginal MAP: objective 130439.72, max |gradient| 4.9e-05
[초3] 3119 students x 959 items, missing rate 0.908
2PL marginal MAP: objective 127525.58, max |gradient| 7.7e-05
[초4] 3494 students x 979 items, missing rate 0.916
2PL marginal MAP: objective 131268.90, max |gradient| 2.8e-05
[초5] 2940 students x 1229 items, missing rate 0.918
2PL marginal MAP: objective 133921.58, max |gradient| 9.9e-05
[초6] 2643 students x 1160 items, missing rate 0.907
2PL marginal MAP: objective 117104.25, max |gradient| 7.5e-05
[중1] 2069 students x 1272 items, missing rate 0.885
2PL marginal MAP: objective 159913.71, max |gradient| 4.9e-05
[중2] 1830 students x 985 items, missing rate 0.859
2PL marginal MAP: objective 133973.82, max |gradient| 6.3e-05
[중3] 772 students x 431 items, missing rate 0.644
2PL marginal MAP: objective 61498.04, max 